In [9]:
import pandas as pd

In [10]:
gender_df_og = pd.read_excel("2020_gender_gen_election.xlsx")
age_df_og = pd.read_excel("2020_age_gen_election.xlsx")
race_df_og = pd.read_excel("2020_race_gen_election.xlsx")
total_df_og = pd.read_excel("2020_total_gen_election.xlsx")

In [ ]:
# run this cell or the above one (depending on your environment)
gender_df_og = pd.read_excel("../data_storage/2020_gender_gen_election.xlsx")
age_df_og = pd.read_excel("../data_storage/2020_age_gen_election.xlsx")
race_df_og = pd.read_excel("../data_storage/2020_race_gen_election.xlsx")
total_df_og = pd.read_excel("../data_storage/2020_total_gen_election.xlsx")

In [11]:
gender_df = gender_df_og.copy()
age_df = age_df_og.copy()
race_df = race_df_og.copy()
total_df = total_df_og.copy()

In [15]:
total_df.columns.to_list()

['County',
 'Registered Voters',
 'Total Ballots Cast',
 'Straight Party Democrat',
 'Straight Party Republican',
 'Absentee Ballots',
 'Turnout as a Percentage of Registered Voters',
 'Absentee as Percentage of Total Ballots']

In [16]:
# clean headers
for df in [gender_df, age_df, race_df, total_df]:
    df.columns.str.strip().str.replace("\n"," ")

gender_df = gender_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Gender"})
age_df = age_df.rename(columns={"Total\nBallots Cast": "Total Ballots Cast - Age"})
race_df = race_df.rename(columns={"Total\nBallots Cast": "Total Ballots Cast - Race"})
total_df = total_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Total"})

In [17]:
for df in [gender_df, age_df, race_df, total_df]:
    df["County"] = df["County"].str.strip()
    df["County"] = df["County"].str.replace("\n"," ")
    df["County"] = df["County"].str.title()

In [18]:
# join the three dfs
merged = gender_df.merge(age_df, on="County").merge(race_df, on="County")

In [19]:
merged.columns.tolist()

['County',
 'Total Ballots Cast - Gender',
 'Total Female',
 'Total Male',
 'Total Not Identified',
 'Total Ballots Cast - Age',
 'Age 18-29',
 'Age 30-39',
 'Age 40-49',
 'Age 50-59',
 'Age 60-69',
 'Age 70-79',
 'Age 80-89',
 'Age 90-99',
 'Age 100+',
 'Total Ballots Cast - Race',
 'Total\nAsian (A)',
 'Total\nAmerican Indian (AI)',
 'Total\nBlack (B)',
 'Total Federally-Registered\n(may be of any race) (F)',
 'Total\nHispanic (H)',
 'Total\nKorean (K)',
 'Total\nOther (O)',
 'Total\nNot Identified (U)',
 'Total\nWhite (W)']

In [20]:
merged["equal"] = (
    (merged["Total Ballots Cast - Gender"] == merged["Total Ballots Cast - Age"]) &
     (merged["Total Ballots Cast - Gender"] == merged["Total Ballots Cast - Race"])
)

mismatches = merged[~merged["equal"]]

In [21]:
print(mismatches)
# output: empty dataframe. means there are no mismatches in total ballots cast among the three dfs. 

Empty DataFrame
Columns: [County, Total Ballots Cast - Gender, Total Female, Total Male, Total Not Identified, Total Ballots Cast - Age, Age 18-29, Age 30-39, Age 40-49, Age 50-59, Age 60-69, Age 70-79, Age 80-89, Age 90-99, Age 100+, Total Ballots Cast - Race, Total
Asian (A), Total
American Indian (AI), Total
Black (B), Total Federally-Registered
(may be of any race) (F), Total
Hispanic (H), Total
Korean (K), Total
Other (O), Total
Not Identified (U), Total
White (W), equal]
Index: []

[0 rows x 26 columns]


In [22]:
compare = merged.merge(total_df, on="County")

compare.head()

,County,Total Ballots Cast - Gender,Total Female,Total Male,Total Not Identified,Total Ballots Cast - Age,Age 18-29,Age 30-39,Age 40-49,Age 50-59,...,Total\nNot Identified (U),Total\nWhite (W),equal,Registered Voters,Total Ballots Cast - Total,Straight Party Democrat,Straight Party Republican,Absentee Ballots,Turnout as a Percentage of Registered Voters,Absentee as Percentage of Total Ballots
0,Autauga,27681,15245,12310,126,27681,3701,4051,4619,5512,...,120,21885,True,43482,27813,5055,12916,3494,0.64,0.126
1,Baldwin,110922,60724,50049,149,110922,10689,13025,15928,20071,...,344,101146,True,176642,110214,13483,57259,12903,0.624,0.117
2,Barbour,10551,6059,4483,9,10551,1092,1218,1351,1964,...,24,6110,True,17819,10560,3989,3470,1296,0.593,0.123
3,Bibb,8797,4801,3989,7,8797,1320,1231,1393,1772,...,14,7300,True,14992,9630,1585,5360,577,0.642,0.06
4,Blount,27694,14749,12935,10,27694,3774,3705,4237,5269,...,43,26645,True,41865,27665,1602,17450,1711,0.661,0.062


In [23]:
# get the difference between total ballots cast and total ballots cast gender (age, and race - remember the last three have the same values)
compare["difference"] = compare["Total Ballots Cast - Total"] - compare["Total Ballots Cast - Gender"]

In [24]:
compare["abs_difference"] = compare["difference"].abs()

In [25]:
compare["difference_percent"] = (compare["abs_difference"]/compare["Registered Voters"])

In [ ]:
# the highest difference_percent is 0.073507 so the difference in values between total ballots cast for gender, age, and total aren't too different from the total_df values for total ballots cast
compare.sort_values("difference_percent", ascending=False)[["County", "Total Ballots Cast - Gender", "Total Ballots Cast - Age", "Total Ballots Cast - Race", "Total Ballots Cast - Total", "Registered Voters", "difference_percent", "difference", "abs_difference"]].head()

,County,Total Ballots Cast - Gender,Total Ballots Cast - Age,Total Ballots Cast - Race,Total Ballots Cast - Total,Registered Voters,difference_percent,difference,abs_difference
17,Conecuh,5709,5709,5709,6472,10380,0.073507,763,763
3,Bibb,8797,8797,8797,9630,14992,0.055563,833,833
54,Pike,12778,12778,12778,13895,23585,0.047361,1117,1117
35,Jackson,23805,23805,23805,22270,39525,0.038836,-1535,1535
50,Montgomery,94347,94347,94347,99326,165228,0.030134,4979,4979


In [16]:
compare.columns.tolist()

['County',
 'Total Ballots Cast - Gender',
 'Total Female',
 'Total Male',
 'Total Not Identified',
 'Total Ballots Cast - Age',
 'Age 18-29',
 'Age 30-39',
 'Age 40-49',
 'Age 50-59',
 'Age 60-69',
 'Age 70-79',
 'Age 80-89',
 'Age 90-99',
 'Age 100+',
 'Total Ballots Cast - Race',
 'Total Asian (A)',
 'Total American Indian (AI)',
 'Total Black (B)',
 'Total Federally- Registered (may be of any race) (F)',
 'Total Hispanic (H)',
 'Total Korean (K)',
 'Total Other (O)',
 'Total White (W)',
 'Total Not Identified (U)',
 'equal',
 'Registered Voters',
 'Total Ballots Cast - Total',
 'Democrat Ballots',
 'Republican Ballots',
 'Absentee Ballots',
 'Turnout as a\nPercentage of Registered Voters',
 'Absentee as\nPercentage of Total Ballots',
 'difference',
 'abs_difference',
 'difference_percent']

In [17]:
compare

,County,Total Ballots Cast - Gender,Total Female,Total Male,Total Not Identified,Total Ballots Cast - Age,Age 18-29,Age 30-39,Age 40-49,Age 50-59,...,Registered Voters,Total Ballots Cast - Total,Democrat Ballots,Republican Ballots,Absentee Ballots,Turnout as a\nPercentage of Registered Voters,Absentee as\nPercentage of Total Ballots,difference,abs_difference,difference_percent
0,Autauga,28401,15750,12581,82,28401,3600,3991,4631,5223,...,46291,28388,4960,13269,1670,0.6133,0.0588,-13,13,0.000281
1,Baldwin,122501,66932,55452,117,122501,11800,13847,17274,20669,...,207643,122542,14334,69308,7771,0.5902,0.0634,41,41,0.000197
2,Barbour,9901,5646,4236,19,9901,970,949,1206,1670,...,17666,9919,3409,2419,593,0.5615,0.0598,18,18,0.001019
3,Bibb,9283,5017,4264,2,9283,1275,1211,1323,1737,...,15152,9278,1245,4628,351,0.6123,0.0378,-5,5,0.000330
4,Blount,28255,15026,13213,16,28255,3914,3800,4195,5005,...,43601,28138,1534,18524,965,0.6454,0.0343,-117,117,0.002683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,Tuscaloosa,85380,48497,36790,93,85380,13505,12401,13533,14310,...,154607,85509,24077,35350,4264,0.5531,0.0499,129,129,0.000834
62,Walker,29843,16062,13749,32,29843,3726,3792,4129,5180,...,49215,29888,2777,17775,859,0.6073,0.0287,45,45,0.000914
63,Washington,8463,4605,3855,3,8463,1047,989,1125,1458,...,13788,8489,1415,4204,447,0.6157,0.0527,26,26,0.001886
64,Wilcox,5268,3028,2240,0,5268,597,561,701,929,...,8557,5287,2827,1036,693,0.6179,0.1311,19,19,0.002220
